## Evaluation, Baselines, Traditional ML

In [1]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [2]:
LITE_MODE = True

In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


## Random Baseline

This is called a random baseline because it predicts a random price between 1 and 1000.

In [4]:
def random_pricer(item):
    return random.randrange(1,1000)

In [5]:
print(len(train))
print(len(val))
print(len(test))
print(test[0])

20000
1000
1000
title='Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal' category='Musical_Instruments' price=219.0 full=None weight=2.0 summary='Title: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.' prompt=None id=None


In [6]:
random.seed(42)
evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$436 $1 $29 $690 $252 $21 $85 $72 $719 $225 $20 $380 $894 $505 $11 $572 $354 $17 $179 $23 $90 $115 $433 $442 $304 $122 $291 $714 $567 $639 $539 $370 $66 $380 $489 $534 $769 $835 $207 $740 $626 $84 $680 $178 $129 $260 $142 $189 $836 $580 $310 $25 $380 $270 $47 $234 $861 $313 $417 $259 $591 $33 $657 $361 $79 $38 $757 $500 $263 $5 $534 $284 $570 $625 $584 $871 $759 $361 $575 $178 $602 $60 $17 $579 $207 $732 $115 $224 $756 $193 $866 $9 $370 $250 $456 $423 $821 $217 $103 $195 $264 $98 $650 $135 $470 $842 $675 $264 $43 $325 $591 $138 $516 $619 $56 $2 $449 $369 $221 $845 $640 $616 $501 $59 $502 $273 $844 $688 $616 $81 $164 $705 $52 $795 $259 $396 $70 $2 $89 $798 $902 $331 $818 $716 $129 $186 $627 $2 $141 $873 $918 $553 $423 $7 $253 $136 $204 $707 $255 $502 $19 $739 $557 $426 $80 $575 $39 $321 $185 $132 $497 $484 $326 $751 $53 $733 $85 $64 $549 $71 $158 $672 $133 $362 $16 $373 $258 $544 $420 $515 $223 $944 $532 $743 $866 $68 $527 $459 $67 $673 

## Constant Baseline

In [7]:
training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

140.347293


In [8]:
evaluate(constant_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$79 $24 $85 $70 $110 $90 $4 $75 $105 $190 $573 $239 $120 $86 $61 $107 $61 $90 $70 $21 $6 $16 $55 $35 $192 $312 $355 $120 $42 $60 $120 $80 $20 $60 $25 $679 $81 $84 $73 $102 $59 $60 $105 $115 $80 $115 $122 $109 $5 $62 $105 $10 $335 $21 $87 $6 $133 $100 $62 $128 $96 $62 $49 $30 $488 $50 $100 $305 $15 $64 $109 $123 $140 $121 $90 $104 $16 $130 $123 $122 $20 $128 $110 $41 $113 $80 $42 $166 $20 $95 $118 $45 $120 $105 $132 $88 $106 $17 $130 $435 $40 $23 $103 $1 $109 $22 $115 $260 $109 $159 $80 $174 $109 $12 $55 $30 $116 $120 $84 $37 $124 $51 $70 $26 $60 $80 $120 $41 $39 $1 $69 $3 $55 $110 $75 $125 $65 $70 $12 $2 $76 $110 $60 $121 $54 $108 $95 $370 $125 $107 $121 $34 $93 $0 $121 $139 $91 $84 $179 $90 $149 $114 $98 $127 $700 $117 $307 $90 $100 $130 $115 $118 $280 $118 $39 $9 $112 $47 $46 $47 $514 $115 $160 $108 $90 $118 $7 $73 $80 $114 $105 $89 $105 $2 $40 $60 $30 $140 $89 $114 

## Predicting error from each ML model

## Linear Regression

Linear Regression is a fundamental supervised learning algorithm used to model the relationship between 

a dependent variable(y) and one or more independent variables(x). 

In [9]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [10]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [11]:
np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

weight: 3.5687444142553026
weight_unknown: 20.90175988023756
text_length: 0.20343123739151991
Intercept: 40.91238780791434
Mean Squared Error: 20096.925335647466
R-squared Score: 0.1609102308229894


In [12]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [13]:
evaluate(linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$79 $51 $62 $66 $71 $50 $9 $50 $82 $191 $574 $233 $113 $58 $28 $97 $50 $62 $39 $14 $18 $14 $75 $11 $193 $318 $359 $91 $35 $36 $104 $78 $27 $16 $79 $676 $65 $66 $70 $67 $64 $36 $61 $66 $104 $93 $104 $83 $25 $60 $84 $5 $347 $2 $79 $20 $110 $83 $23 $123 $111 $49 $19 $3 $241 $17 $82 $261 $6 $68 $85 $105 $155 $89 $91 $80 $3 $98 $99 $88 $72 $86 $86 $8 $84 $43 $81 $176 $62 $60 $88 $42 $80 $71 $91 $117 $78 $8 $146 $418 $33 $7 $106 $21 $122 $1 $83 $298 $104 $198 $62 $154 $106 $59 $71 $53 $103 $94 $82 $14 $99 $43 $51 $51 $127 $47 $86 $102 $72 $16 $53 $9 $47 $72 $45 $84 $81 $40 $9 $11 $39 $132 $55 $85 $3 $71 $71 $364 $5 $72 $93 $26 $53 $21 $90 $119 $103 $40 $43 $79 $160 $92 $67 $116 $725 $100 $279 $74 $84 $101 $86 $100 $244 $78 $139 $58 $95 $7 $50 $38 $289 $117 $38 $94 $75 $96 $6 $77 $57 $79 $79 $78 $108 $21 $14 $97 $50 $109 $90 $114 

CountVectorizer and Bag of Words using Linear Regression (NLP+LR)

In [14]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [15]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [16]:
# Here are the 1,000 most common words that it picked, not including "stop words":
selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1080])

Number of selected words: 2000
Selected words: ['items' 'jack' 'jacket' 'jeep' 'jigsaw' 'joint' 'joints' 'keeping'
 'keeps' 'key' 'keyboard' 'keypad' 'keys' 'kg' 'khz' 'kia' 'kickstand'
 'kids' 'king' 'kingston' 'kit' 'kitchen' 'knife' 'knob' 'knobs' 'kohler'
 'kraft' 'label' 'labels' 'laminated' 'lamp' 'lamps' 'lantern' 'lanyard'
 'laptop' 'laptops' 'large' 'laser' 'laserjet' 'lasting' 'latch' 'latex'
 'latitude' 'layer' 'layout' 'lb' 'lbs' 'lcd' 'lead' 'leak' 'learning'
 'leather' 'led' 'leds' 'left' 'lego' 'legs' 'length' 'lenovo' 'lens'
 'lenses' 'lets' 'letter' 'level' 'levels' 'lever' 'lexus' 'lg' 'li'
 'licensed' 'lid' 'life' 'lifespan' 'lifetime' 'lift' 'light' 'lighting'
 'lights' 'lightweight' 'like']


In [17]:
regressor = LinearRegression()
regressor.fit(X, prices)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [18]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [19]:
evaluate(natural_language_linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$86 $120 $55 $70 $191 $230 $24 $41 $8 $65 $480 $188 $61 $132 $19 $139 $64 $31 $25 $27 $39 $86 $51 $20 $318 $360 $149 $36 $38 $32 $75 $58 $3 $26 $120 $338 $10 $106 $142 $73 $151 $110 $88 $9 $89 $42 $74 $72 $14 $35 $41 $49 $154 $42 $78 $167 $111 $194 $15 $12 $40 $3 $55 $21 $426 $35 $26 $191 $11 $224 $24 $46 $92 $111 $19 $49 $140 $11 $20 $125 $7 $37 $71 $43 $61 $103 $5 $149 $52 $255 $30 $142 $84 $41 $52 $133 $64 $56 $213 $108 $88 $57 $37 $35 $3 $51 $14 $162 $31 $71 $15 $79 $206 $13 $33 $123 $119 $31 $49 $19 $16 $159 $8 $27 $65 $24 $20 $45 $58 $59 $5 $63 $181 $14 $26 $47 $1 $151 $99 $104 $49 $141 $5 $203 $139 $74 $19 $317 $19 $22 $19 $263 $30 $37 $43 $56 $163 $16 $102 $13 $63 $26 $42 $13 $387 $9 $85 $50 $2 $10 $10 $61 $318 $65 $69 $82 $8 $74 $0 $120 $386 $11 $58 $21 $107 $39 $5 $67 $150 $27 $38 $52 $33 $36 $32 $59 $45 $28 $15 $36 

## Random Forest model
A Decision Tree works like a series of questions.

The problem is that one decision tree can make mistakes and overfit the training data.

Random Forest is a supervised machine learning algorithm that combines multiple decision trees to 

generate a single, more accurate prediction.

In this project

After CountVectorizer, the product summary becomes numbers:

Ex:-

Laptop = 1

RAM = 1

SSD = 1

Gaming = 0

TV = 0

Random Forest uses these word counts as features and learns patterns such as:

Laptop + SSD + RAM → Higher Price

Cable + USB → Lower Price

TV + LED + 4K → Medium/High Price



In [20]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [22]:
evaluate(random_forest, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$67 $67 $8 $10 $112 $139 $74 $56 $27 $266 $517 $288 $40 $76 $11 $17 $50 $106 $33 $50 $22 $9 $9 $30 $148 $316 $161 $124 $30 $49 $54 $80 $86 $17 $21 $367 $48 $28 $145 $82 $150 $42 $18 $42 $146 $28 $48 $40 $59 $11 $11 $29 $236 $29 $314 $41 $36 $157 $7 $47 $107 $4 $61 $6 $460 $3 $14 $179 $28 $129 $12 $24 $69 $128 $4 $97 $107 $17 $33 $73 $60 $70 $23 $48 $14 $31 $14 $101 $5 $101 $33 $164 $1 $82 $57 $64 $2 $47 $138 $274 $44 $6 $10 $78 $34 $29 $116 $205 $20 $176 $37 $68 $63 $29 $69 $22 $76 $11 $216 $25 $27 $40 $60 $37 $40 $28 $43 $46 $105 $2 $24 $48 $68 $17 $37 $53 $30 $11 $75 $32 $12 $151 $4 $103 $106 $66 $36 $325 $79 $4 $21 $99 $48 $75 $57 $80 $112 $39 $35 $10 $6 $4 $3 $27 $400 $46 $271 $31 $6 $35 $21 $6 $213 $25 $9 $65 $47 $5 $42 $4 $341 $16 $83 $87 $161 $57 $50 $83 $2 $39 $51 $52 $25 $17 $6 $21 $68 $36 $5 $17 

### XG Boost

XGBoost is similar to Random Forest because both use decision trees.

The difference is how they build those trees.

Random Forest

Tree 1

Tree 2

Tree 3

...

Tree 100

All trees are built independently.

Each tree gives a prediction and the final answer is: Average of all trees

XGBoost

XGBoost builds trees one after another.

Tree 1

  ↓

Find mistakes

  ↓

Tree 2 fixes mistakes

  ↓

Find remaining mistakes

  ↓

Tree 3 fixes those mistakes

  ↓

...

In [23]:
import xgboost as xgb

In [24]:
np.random.seed(42)
xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [25]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [26]:
evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$82 $81 $30 $16 $107 $160 $39 $3 $16 $21 $601 $245 $107 $107 $0 $17 $55 $13 $15 $15 $11 $25 $5 $68 $234 $321 $95 $31 $40 $28 $84 $55 $9 $9 $54 $124 $24 $69 $164 $68 $154 $88 $63 $8 $118 $113 $37 $57 $43 $8 $30 $37 $126 $8 $236 $95 $61 $193 $2 $41 $72 $8 $71 $35 $444 $33 $46 $208 $50 $100 $2 $43 $92 $106 $13 $200 $222 $10 $34 $117 $22 $95 $87 $40 $19 $50 $63 $124 $26 $226 $34 $196 $27 $3 $25 $93 $124 $26 $199 $168 $114 $40 $23 $61 $5 $91 $175 $207 $31 $66 $39 $64 $72 $54 $79 $140 $129 $29 $136 $42 $45 $98 $35 $33 $135 $4 $16 $97 $179 $99 $38 $52 $141 $35 $9 $70 $74 $48 $164 $93 $2 $148 $55 $89 $82 $96 $36 $260 $11 $5 $19 $163 $46 $60 $25 $96 $117 $16 $21 $4 $74 $21 $8 $13 $397 $2 $147 $50 $15 $11 $22 $26 $209 $32 $31 $47 $50 $32 $37 $58 $423 $12 $137 $30 $113 $59 $61 $45 $65 $30 $47 $20 $16 $17 $4 $71 $57 $41 $40 $26 

## In Day 3, the main idea is to compare:

Actual Price

      vs

Predicted Price

The difference between them is the error.

## Mean Absolute Error (MAE)

Average of all prediction errors

Lower is better.

## Mean Squared Error (MSE)

(actual - predicted)²

Large mistakes are penalized more.

## R² Score

How well the model explains the prices

Higher is better.